# Scaling Laws: How Big Should the Model Be? How Much Data? How Much Money?

> **Background**: you want to train an LLM. Your boss gives you a $1M GPU budget and asks:
> "With this budget, how big should the model be? How much data should we use? How good can it get?"
>
> Goal for this part: **understand the three paradigm shifts of scaling laws, and learn to estimate training compute and VRAM.**

There are only two core questions:
1. **Given a compute budget, what is the best model-vs-data tradeoff?** -> scaling laws
2. **To actually train a model, how many GPUs and how much money do we need?** -> FLOPs / VRAM estimation

We will use small, high-school-level examples and hand calculations so every step is concrete.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part A: Scaling laws

### 1. What is a power law? Understand it via savings interest

Before scaling laws, you must understand the idea of a **power law**.

#### 1.1 Linear improvement vs power-law improvement

Suppose your saved money doubles each month:

- **Linear**: every doubling adds a fixed amount
  - 100 -> gain 10
  - 200 -> gain 20 (+10)
  - 400 -> gain 30 (+10)

- **Power law**: every doubling improves by a fixed *ratio*
  - 100 -> gain 10
  - 200 -> gain 18 (+80%)
  - 400 -> gain 32.4 (+80%)

LLM training loss often follows a power law: **when model size doubles, loss drops by roughly a fixed ratio.**

A common form is:
`Loss(N) ≈ a × N^(-b) + c`

Where:
- `N` = number of parameters
- `a, b, c` = constants fit from experiments
- `N^(-b)` gets smaller as `N` grows
- `c` is the irreducible loss floor (you cannot go below it by scaling N)


In [2]:
# A tiny manual demo of diminishing returns in a power law
print("=== Power law diminishing returns: loss vs model size ===")
print()

# Simulate: loss = a * N^(-b) + c
# Use small numbers to make the trend obvious.
a, b_pow, c = 5.0, 0.05, 2.0

# Model sizes from 1M params to 100B params
sizes = [1, 10, 100, 1000, 10000, 100000]  # in millions of params
labels = ["1M", "10M", "100M", "1B", "10B", "100B"]

print(f"Formula: Loss = {a} * N^(-{b_pow}) + {c}")
print()
print(f"{'Model size':>10s} {'Loss':>8s} {'vs previous':>14s}")
print("-" * 36)

prev_loss = None
for size, label in zip(sizes, labels):
    N = size * 1e6
    loss = a * N ** (-b_pow) + c

    if prev_loss is not None:
        improvement = (prev_loss - loss) / prev_loss * 100
        print(f"{label:>10s}  {loss:>6.2f}  down {improvement:>6.1f}%")
    else:
        print(f"{label:>10s}  {loss:>6.2f}  --")
    prev_loss = loss

print()
print("Key observation:")
print("  1. Scaling the model up does reduce loss")
print("  2. But each extra 10x gives a smaller gain (diminishing returns)")
print("  3. The gain from 1B->10B is smaller than 1M->10M")
print()
print("That is the core message of a power law: bigger helps, but it becomes less worth it.")


=== Power law diminishing returns: loss vs model size ===

Formula: Loss = 5.0 * N^(-0.05) + 2.0

Model size     Loss    vs previous
------------------------------------
        1M    4.51  --
       10M    4.23  down    6.0%
      100M    3.99  down    5.7%
        1B    3.77  down    5.4%
       10B    3.58  down    5.1%
      100B    3.41  down    4.8%

Key observation:
  1. Scaling the model up does reduce loss
  2. But each extra 10x gives a smaller gain (diminishing returns)
  3. The gain from 1B->10B is smaller than 1M->10M

That is the core message of a power law: bigger helps, but it becomes less worth it.


### 2. Kaplan scaling law (OpenAI, 2020): "scale the model first"

In 2020, OpenAI ran many experiments to answer:
Given a fixed compute budget `C` (e.g. `1e20` FLOPs), what is the best choice of model size `N` and data size `D`?

#### 2.1 Key finding

The optimal scaling they reported looked like:

```
N_opt ∝ C^0.73    (model size grows fast with compute)
D_opt ∝ C^0.27    (data grows slowly with compute)
```

Because `0.73 > 0.27`, **model size grows much faster than data**.

Plain English: when you double the compute budget, you should spend most of the extra budget on a larger model, and a smaller part on more data.

#### 2.2 Feel it with numbers


In [3]:
# Kaplan-style scaling: for a fixed compute budget, what is the "optimal" split?
print("=== Kaplan scaling laws: given a compute budget, what is the optimal allocation? ===")
print()

# Reference point: assume at C=1e20 FLOPs, N=1B params, D=20B tokens
N_ref = 1e9
D_ref = 20e9
C_ref = 1e20

print(
    f"Reference: C={C_ref/1e20:.0f}x10^20 FLOPs -> N={N_ref/1e9:.0f}B params, D={D_ref/1e9:.0f}B tokens"
)
print()

# Compute budgets to compare
budgets = [1e20, 2e20, 4e20, 8e20, 1e21, 1e22]
budget_labels = ["1x", "2x", "4x", "8x", "10x", "100x"]

print(f"{'Compute':>12s}  {'Model size (Kaplan)':>20s}  {'Data size (Kaplan)':>20s}  {'D/N':>8s}")
print("-" * 72)

for C, label in zip(budgets, budget_labels):
    ratio = C / C_ref
    N_opt = N_ref * (ratio ** 0.73)
    D_opt = D_ref * (ratio ** 0.27)
    dn_ratio = D_opt / N_opt

    n_str = f"{N_opt/1e9:.1f}B" if N_opt >= 1e9 else f"{N_opt/1e6:.0f}M"
    d_str = f"{D_opt/1e9:.1f}B tokens"
    print(f"{label:>12s}  {n_str:>20s}  {d_str:>20s}  {dn_ratio:>6.1f}x")

print()
print("Kaplan's implication:")
print("  - With more compute, grow the model a lot faster than the dataset")
print("  - D/N becomes smaller (you can 'reuse' data more with a bigger model)")


=== Kaplan scaling laws: given a compute budget, what is the optimal allocation? ===

Reference: C=1x10^20 FLOPs -> N=1B params, D=20B tokens

     Compute   Model size (Kaplan)    Data size (Kaplan)       D/N
------------------------------------------------------------------------
          1x                  1.0B          20.0B tokens    20.0x
          2x                  1.7B          24.1B tokens    14.5x
          4x                  2.8B          29.1B tokens    10.6x
          8x                  4.6B          35.1B tokens     7.7x
         10x                  5.4B          37.2B tokens     6.9x
        100x                 28.8B          69.3B tokens     2.4x

Kaplan's implication:
  - With more compute, grow the model a lot faster than the dataset
  - D/N becomes smaller (you can 'reuse' data more with a bigger model)


### 3. Chinchilla scaling law (DeepMind, 2022): "model and data both matter"

#### 3.1 Why Kaplan was "wrong"

DeepMind repeated the experiments and found that in OpenAI's earlier setting, when compute increased, **data did not increase enough** (high-quality data was limited at the time).
So "scale the model first" was a conclusion under a *data-starved* regime.

#### 3.2 Chinchilla's conclusion

DeepMind swept many (N, D) combinations for each compute budget and found the true optimum:

```
N_opt ∝ C^0.5
D_opt ∝ C^0.5
-> model and data are equally important
-> D_opt ≈ 20 × N_opt   (about 20 tokens per parameter)
```

#### 3.3 Under the same compute budget, Kaplan vs Chinchilla choose different points


In [4]:
# Kaplan vs Chinchilla under the same compute budget
print("=== Kaplan vs Chinchilla: different recipes under the same compute ===")
print()

C = 1e23  # roughly GPT-3 scale
ratio = C / C_ref

# Kaplan exponents
N_k = N_ref * (ratio ** 0.73)
D_k = D_ref * (ratio ** 0.27)

# Chinchilla exponents (roughly ~sqrt for both)
N_c = N_ref * (ratio ** 0.5)
D_c = D_ref * (ratio ** 0.5)

print(f"Compute budget: {C/1e23:.0f} x 10^23 FLOPs")
print()
print(f"{'':>20s}  {'Model size':>12s}  {'Data size':>14s}  {'D/N':>10s}")
print("-" * 62)
print(f"{'Kaplan says':>20s}  {N_k/1e9:>8.1f}B    {D_k/1e9:>9.1f}B tokens  {D_k/N_k:>8.1f}x")
print(f"{'Chinchilla says':>20s}  {N_c/1e9:>8.1f}B    {D_c/1e9:>9.1f}B tokens  {D_c/N_c:>8.1f}x")
print()

print("Comparison:")
print(f"  Kaplan: model is {N_k/N_c:.1f}x larger, but data is only {D_k/D_c:.1f}x")
print("  Chinchilla: smaller model, more data")
print()
print("Empirically, Chinchilla-style (more data) often wins at fixed compute.")


=== Kaplan vs Chinchilla: different recipes under the same compute ===

Compute budget: 1 x 10^23 FLOPs

                        Model size       Data size         D/N
--------------------------------------------------------------
         Kaplan says     154.9B        129.1B tokens       0.8x
     Chinchilla says      31.6B        632.5B tokens      20.0x

Comparison:
  Kaplan: model is 4.9x larger, but data is only 0.2x
  Chinchilla: smaller model, more data

Empirically, Chinchilla-style (more data) often wins at fixed compute.


#### 3.4 The shocking example

DeepMind trained two models under roughly the same compute budget:

```
Gopher:      280B params, 300B tokens   (Kaplan-style)
Chinchilla:   70B params, 1.4T tokens  (Chinchilla-style)
  same total compute

Result: Chinchilla (70B) >> Gopher (280B)
        -> 4x smaller model, but 4.7x more data, and it wins
```

The punchline: many large models in that era were likely **under-trained on data**.


### 4. Post-Chinchilla: "overtraining" can be even better

Chinchilla says D/N ~= 20 is optimal, but 2023-2024 practice often goes far beyond that.

#### 4.1 Why? Because Chinchilla optimizes *training cost*

Chinchilla's "optimal" means: **under a fixed training compute budget, minimize loss.**

But in deployment, **inference cost dominates**. Serving millions of users can cost far more than one-time training.

So the strategy becomes: **train a smaller model, feed it a lot more data, and get strong performance with cheaper inference.**

Example:

```
LLaMA 3 8B:
  Chinchilla would suggest ~ 8B * 20 = 160B tokens
  In practice: ~15T tokens (about 94x the Chinchilla point)
  Result: much stronger than an 8B model trained at the Chinchilla optimum
```

This is the train-vs-infer tradeoff: pay more once in training to save a lot in inference.


In [5]:
# D/N ratio for some real models (very rough numbers)
print("=== D/N ratio for some well-known models ===")
print()

real_models = [
    ("Chinchilla (optimal)", 70, 1400, "20x"),
    ("LLaMA 7B", 7, 1000, "143x"),
    ("LLaMA 2 7B", 7, 2000, "286x"),
    ("LLaMA 3 8B", 8, 15000, "1875x"),
    ("DeepSeek-V2", 236, 8100, "34x (active: ~386x)"),
]

print(f"{'Model':<20s} {'Params':>8s} {'Data':>12s} {'D/N':>14s}")
print("-" * 58)
for name, params_b, tokens_b, dn in real_models:
    print(f"{name:<20s} {params_b:>5}B   {tokens_b:>6}B tokens  {dn:>14s}")

print()
print("Observation: post-2023 models often use D/N far above 20x.")
print("Reason: smaller models are much cheaper at inference, so spending more on data can pay off.")


=== D/N ratio for some well-known models ===

Model                  Params         Data            D/N
----------------------------------------------------------
Chinchilla (optimal)    70B     1400B tokens             20x
LLaMA 7B                 7B     1000B tokens            143x
LLaMA 2 7B               7B     2000B tokens            286x
LLaMA 3 8B               8B    15000B tokens           1875x
DeepSeek-V2            236B     8100B tokens  34x (active: ~386x)

Observation: post-2023 models often use D/N far above 20x.
Reason: smaller models are much cheaper at inference, so spending more on data can pay off.


### 5. µP (Maximal Update Parameterization): how to tune hyperparams on small models

#### 5.1 The problem

Before training a 70B model, you want to tune hyperparameters (learning rate, init, etc.).
But you cannot grid-search on 70B; one run can cost hundreds of thousands.

A common workflow is: tune on a 10M model, then reuse the hyperparams on 70B.

**Problem**: it often does not transfer. The best LR on 10M can be unstable on 70B.

#### 5.2 Why?

Different model sizes have different activation and gradient scales.
An LR that is right for 10M can be too small (no learning) or too large (diverges) for 70B.

#### 5.3 How µP helps

µP rescales initialization variance and learning rates so that **early training dynamics match across model sizes**.

Plain English: µP makes 10M and 70B models "feel" the same to train, so hyperparams found on small scale can transfer.

```
Without µP: tune lr=0.01 on 10M -> move to 70B -> unstable, retune
With µP:    tune lr=0.01 on 10M -> move to 70B -> works
```

This can save huge human and compute cost.


---

## Part B: Resource estimation

### 6. FLOPs estimation: how much compute does training need?

#### 6.1 Core formula

```
C ≈ 6 × P × D
```

Where:
- `C` = total compute (FLOPs)
- `P` = number of parameters
- `D` = number of training tokens

#### 6.2 Why the factor 6? Do it by hand

Training one token needs forward + backward.

Forward pass:
- dominated by matrix multiplies
- roughly ~2P FLOPs per token

Backward pass:
- computing gradients is roughly ~2x forward
- ~4P FLOPs per token

Total: `2P + 4P = 6P` FLOPs per token.
So for D tokens: `C = 6PD`.

#### 6.3 Compute a concrete example


In [6]:
# FLOPs estimation: step-by-step
print("=== FLOPs estimation (manual) ===")
print()

# Question: train a LLaMA 7B on 1T tokens, how many FLOPs?
P = 7e9
D = 1e12

print(f"Model params P = {P/1e9:.0f}B")
print(f"Training tokens D = {D/1e12:.0f}T")
print()

# Step 1: FLOPs per token
flops_per_token = 6 * P
print(f"Step 1 - FLOPs per token = 6 * {P/1e9:.0f}B")
print(f"       = {flops_per_token:.1e} FLOPs/token")
print()

# Step 2: Total FLOPs
total_flops = flops_per_token * D
print(f"Step 2 - Total FLOPs = {flops_per_token:.1e} * {D/1e12:.0f}T")
print(f"       = {total_flops:.1e} FLOPs")
print()

# Convert to a human-readable unit
def format_flops(flops):
    if flops >= 1e24:
        return f"{flops/1e24:.1f} YFLOPs (1e24)"
    if flops >= 1e21:
        return f"{flops/1e21:.1f} ZFLOPs (1e21)"
    if flops >= 1e18:
        return f"{flops/1e18:.1f} EFLOPs (1e18)"
    return f"{flops/1e15:.1f} PFLOPs (1e15)"

print(f"Step 3 - Result: {format_flops(total_flops)}")
print()

# Step 4: GPU time
print("Step 4 - Using an A100 (312 TFLOPS FP16, 50% utilization):")
effective_tflops = 312 * 0.5
flops_per_second = effective_tflops * 1e12
seconds = total_flops / flops_per_second
hours = seconds / 3600
days = hours / 24
print(f"  Effective throughput: {effective_tflops:.0f} TFLOPS")
print(f"  Time needed: {hours:,.0f} hours = {days:.0f} days (single GPU)")
print(f"  With 256 A100s: {hours/256:.0f} hours = {days/256:.1f} days")


=== FLOPs estimation (manual) ===

Model params P = 7B
Training tokens D = 1T

Step 1 - FLOPs per token = 6 * 7B
       = 4.2e+10 FLOPs/token

Step 2 - Total FLOPs = 4.2e+10 * 1T
       = 4.2e+22 FLOPs

Step 3 - Result: 42.0 ZFLOPs (1e21)

Step 4 - Using an A100 (312 TFLOPS FP16, 50% utilization):
  Effective throughput: 156 TFLOPS
  Time needed: 74,786 hours = 3116 days (single GPU)
  With 256 A100s: 292 hours = 12.2 days


In [7]:
# Compare training FLOPs for some major models (very rough)
print("=== Major model training FLOPs (rough overview) ===")
print()

models = [
    ("GPT-3 175B", 175e9, 300e9),
    ("LLaMA 7B", 7e9, 1e12),
    ("LLaMA 65B", 65e9, 1.4e12),
    ("Chinchilla 70B", 70e9, 1.4e12),
    ("LLaMA 2 70B", 70e9, 2e12),
    ("LLaMA 3 8B", 8e9, 15e12),
    ("LLaMA 3 70B", 70e9, 15e12),
    ("DeepSeek-V3 (total)", 671e9, 14.8e12),
]

print(f"{'Model':<20s} {'Params':>8s} {'Data':>14s} {'Total FLOPs':>14s} {'A100 days (256)':>15s}")
print("-" * 80)

for name, params, tokens in models:
    flops = 6 * params * tokens
    effective_tflops = 312 * 0.5
    hours = flops / (effective_tflops * 1e12 * 3600)
    days_256 = hours / 256 / 24

    print(
        f"{name:<20s} {params/1e9:>5.0f}B  {tokens/1e9:>9.0f}B tokens"
        f"  {format_flops(flops):>14s}  {days_256:>15.1f}"
    )

print()
print("Notes:")
print("  - This is only an estimate. In practice there is extra overhead (activations, comms, etc.)")
print("  - 50% utilization is optimistic for large clusters")
print("  - This counts one training run, not failed runs or hyperparameter search")


=== Major model training FLOPs (rough overview) ===

Model                  Params           Data    Total FLOPs A100 days (256)
--------------------------------------------------------------------------------
GPT-3 175B             175B        300B tokens  315.0 ZFLOPs (1e21)             91.3
LLaMA 7B                 7B       1000B tokens  42.0 ZFLOPs (1e21)             12.2
LLaMA 65B               65B       1400B tokens  546.0 ZFLOPs (1e21)            158.2
Chinchilla 70B          70B       1400B tokens  588.0 ZFLOPs (1e21)            170.4
LLaMA 2 70B             70B       2000B tokens  840.0 ZFLOPs (1e21)            243.4
LLaMA 3 8B               8B      15000B tokens  720.0 ZFLOPs (1e21)            208.7
LLaMA 3 70B             70B      15000B tokens  6.3 YFLOPs (1e24)           1825.8
DeepSeek-V3 (total)    671B      14800B tokens  59.6 YFLOPs (1e24)          17268.6

Notes:
  - This is only an estimate. In practice there is extra overhead (activations, comms, etc.)
  - 50% utili

### 7. VRAM estimation: how many GPUs do you need?

#### 7.1 What consumes VRAM during training?

During training, VRAM has several major tenants:

```
+-----------------------------------------+
|              GPU VRAM                   |
+-----------------------------------------+
| 1) Parameters (FP16)           2P bytes |
| 2) Gradients (FP16)            2P bytes |
| 3) AdamW states (FP32)         8P bytes |
|    - m (momentum)              4P bytes |
|    - v (variance)              4P bytes |
| 4) Master params (FP32)        4P bytes |
| 5) Activations                 ~2-10P   |
+-----------------------------------------+
| Total                         ~18-26P   |
+-----------------------------------------+
```

Rule of thumb: **~20 bytes of VRAM per parameter** for training.

#### 7.2 Example: LLaMA 7B


In [8]:
# VRAM estimation: step-by-step
print("=== VRAM estimate: how much VRAM to train LLaMA 7B? ===")
print()

P = 7e9  # 7B params

# Break down the components (very rough)
param_fp16 = 2 * P
grad_fp16 = 2 * P
adam_m = 4 * P
adam_v = 4 * P
param_fp32 = 4 * P

model_optimizer = param_fp16 + grad_fp16 + adam_m + adam_v + param_fp32

print("Model params + optimizer state:")
print(f"  1) Params (FP16):          {param_fp16/1e9:.1f} GB")
print(f"  2) Gradients (FP16):       {grad_fp16/1e9:.1f} GB")
print(f"  3) Adam m (FP32):          {adam_m/1e9:.1f} GB")
print(f"  4) Adam v (FP32):          {adam_v/1e9:.1f} GB")
print(f"  5) FP32 master params:     {param_fp32/1e9:.1f} GB")
print(f"  Subtotal:                  {model_optimizer/1e9:.1f} GB")
print(f"  = {model_optimizer/P:.0f} bytes/param")
print()

# Activations (rough): depends on batch, seq_len, hidden size, layers, etc.
batch_size = 4
seq_len = 4096

print(f"Activations depend on batch={batch_size}, seq_len={seq_len}.")
print(f"A very rough rule: ~2-10P bytes = {2*P/1e9:.0f}-{10*P/1e9:.0f} GB")
print()

activation_low = 2 * P
activation_high = 10 * P

total_low = (model_optimizer + activation_low) / 1e9
total_high = (model_optimizer + activation_high) / 1e9

print(f"Total VRAM: {total_low:.0f} - {total_high:.0f} GB")
print(f"A100 80GB needed: {total_low/80:.0f} - {total_high/80:.0f} GPUs")
print()

print("=== Typical training VRAM needs (rule-of-thumb) ===")
print()
print(f"{'Model':<18s} {'model+opt':>10s} {'Total (w/ act)':>16s} {'A100 80GB':>10s}")
print("-" * 62)

for name, params in [
    ("1.5B model", 1.5e9),
    ("LLaMA 7B", 7e9),
    ("LLaMA 13B", 13e9),
    ("LLaMA 65B", 65e9),
    ("GPT-3 175B", 175e9),
]:
    m_o = 16 * params / 1e9
    total = 20 * params / 1e9
    gpus = total / 80
    print(f"{name:<18s} {m_o:>6.0f} GB   ~{total:>10.0f} GB      {gpus:>8.0f} GPUs")

print()
print("Rule-of-thumb: 1B params ~= 20GB VRAM for training")
print("Inference is much cheaper (~2GB/B) because you don't store gradients/optimizer/activations")


=== VRAM estimate: how much VRAM to train LLaMA 7B? ===

Model params + optimizer state:
  1) Params (FP16):          14.0 GB
  2) Gradients (FP16):       14.0 GB
  3) Adam m (FP32):          28.0 GB
  4) Adam v (FP32):          28.0 GB
  5) FP32 master params:     28.0 GB
  Subtotal:                  112.0 GB
  = 16 bytes/param

Activations depend on batch=4, seq_len=4096.
A very rough rule: ~2-10P bytes = 14-70 GB

Total VRAM: 126 - 182 GB
A100 80GB needed: 2 - 2 GPUs

=== Typical training VRAM needs (rule-of-thumb) ===

Model               model+opt   Total (w/ act)  A100 80GB
--------------------------------------------------------------
1.5B model             24 GB   ~        30 GB             0 GPUs
LLaMA 7B              112 GB   ~       140 GB             2 GPUs
LLaMA 13B             208 GB   ~       260 GB             3 GPUs
LLaMA 65B            1040 GB   ~      1300 GB            16 GPUs
GPT-3 175B           2800 GB   ~      3500 GB            44 GPUs

Rule-of-thumb: 1B params

### 8. Summary of the three paradigm shifts

```
2020 Kaplan (OpenAI):
  "Scale the model; data can be smaller"
  N ∝ C^0.73, D ∝ C^0.27
  -> GPT-3 175B + 300B tokens (D/N=1.7x)

2022 Chinchilla (DeepMind):
  "Model and data are equally important"
  N ∝ C^0.5, D ∝ C^0.5, D/N ≈ 20
  -> Chinchilla 70B + 1.4T tokens (D/N=20x) beats Gopher 280B

2023+ Overtraining (LLaMA 3, etc.):
  "Small model + massive data for cheaper inference"
  D/N >> 20, sometimes > 1000
  -> LLaMA 3 8B + 15T tokens (D/N=1875x)
```

Each shift overturned a piece of "common sense".


---

## Scaling Laws Checklist

Make sure you understand these (in order):

1. [ok] Power law: `Loss ∝ N^(-b)`; diminishing returns (doubling gives smaller and smaller absolute gains)
2. [ok] Kaplan: scale the model first; D grows slower than N
3. [ok] Chinchilla: model and data both matter; D/N ≈ 20
4. [ok] Overtraining: inference cost > training cost -> smaller model + more data can be optimal
5. [ok] µP: lets hyperparams transfer across model sizes; reduces tuning cost
6. [ok] FLOPs: `C ≈ 6PD` (forward ~2P + backward ~4P per token)
7. [ok] VRAM: `M ≈ 20P bytes` (params+grads+Adam+master+acts)
8. [ok] Rule of thumb: 1B params ~ 20GB training VRAM; inference ~2GB/B (no grads/optimizer/acts)

One-sentence summary: scaling laws tell you the best tradeoff (theory), FLOPs/VRAM estimates tell you whether you can afford it (engineering). In 2024, a common strategy is: smaller models + lots of data, optimizing for inference cost.
